# SNP, amino acid, CNV and haplotype frequency analysis

This notebook documents the `Ag3` methods that compute and plot allele/CNV/haplotype frequencies across cohorts of samples: `snp_allele_frequencies`, `snp_allele_frequencies_advanced`, `aa_allele_frequencies`, `aa_allele_frequencies_advanced`, `gene_cnv_frequencies`, `gene_cnv_frequencies_advanced`, `haplotypes_frequencies`, `haplotypes_frequencies_advanced`, `plot_frequencies_heatmap`, `plot_frequencies_time_series`, `plot_frequencies_interactive_map` and `plot_frequencies_map_markers`.

There are two flavours of each frequency-computation method:

- the **simple** form (`snp_allele_frequencies`, `aa_allele_frequencies`, `gene_cnv_frequencies`, `haplotypes_frequencies`) groups samples into cohorts using the `cohorts` parameter (either a predefined cohort scheme name, e.g. `"admin1_year"`, or a dict mapping custom cohort labels to sample queries) and returns a **dataframe** with one column of frequencies per cohort — convenient for a quick table or `plot_frequencies_heatmap`.
- the **advanced** form (`*_advanced` variants) groups samples by taxon, spatial `area_by` and temporal `period_by` columns automatically, and returns an **xarray Dataset** with a `cohorts` dimension carrying rich cohort metadata (lat/lon, period start/end, taxon, size) alongside the frequency data — this is the structure needed by `plot_frequencies_time_series`, `plot_frequencies_interactive_map` and `plot_frequencies_map_markers`.

**Diagram opportunity:** a diagram showing how these methods relate: `snp_allele_frequencies` / `aa_allele_frequencies` / `gene_cnv_frequencies` / `haplotypes_frequencies` (cohort-dict input, dataframe output) versus their `_advanced` counterparts (area/period/taxon input, xarray Dataset output), and which plotting function each output feeds into (dataframe → `plot_frequencies_heatmap`; Dataset → `plot_frequencies_time_series` / `plot_frequencies_interactive_map` / `plot_frequencies_map_markers`).

In [1]:
import malariagen_data

ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `snp_allele_frequencies`

Computes SNP allele frequencies for a gene transcript or genomic region, across a set of cohorts, returning one row per variant allele. Parameters:

- `transcript` / `region`: provide exactly one. `transcript` is a transcript ID (e.g. `"AGAP004707-RD"`, the *Vgsc*/*para* sodium channel gene linked to pyrethroid resistance); `region` is a more general contig/region-string alternative.
- `cohorts` (required): either the name of a predefined cohort scheme (e.g. `"admin1_year"`, grouping by first-level administrative division and year) or a dict mapping custom cohort labels to sample-query strings.
- `sample_query` / `sample_query_options`: further restrict the sample set used to build cohorts, e.g. by taxon.
- `min_cohort_size`: cohorts with fewer samples than this raise a `ValueError` rather than being silently dropped (unlike the `_advanced` variants, which just omit undersized cohorts).
- `site_mask`: restrict to sites passing a given site-filter mask; `None` (default) applies no site filter.
- `sample_sets`: which sample set(s)/release(s) to draw samples from.
- `drop_invariant`: if `True` (default), drop variant alleles never observed in the selected samples.
- `effects`: if `True` (default), annotate each variant with its predicted coding effect (requires `transcript`).
- `include_counts`: if `True`, also include raw allele-count and `nobs` columns alongside the frequency columns.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.

In [2]:
snp_freqs_df = ag3.snp_allele_frequencies(
    transcript="AGAP004707-RD",
    cohorts="admin1_year",
    sample_sets=("AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-BF-C"),
    sample_query="taxon == 'coluzzii'",
)
snp_freqs_df

Load sample metadata: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.08)

Load genome features: ⠹ (0:00:00.17)

Load genome features: ⠸ (0:00:00.30)

...

Load genome features: ⠸ (0:00:00.34)

Load genome features: ⠼ (0:00:00.45)

Compute SNP effects:   0%|          | 0/5242 [00:00<?, ?it/s]

Load genome features: ⠋ (0:00:00.00)

pass_gamb_colu_arab  \
contig position ref_allele alt_allele aa_change                        
2L     2358252  C          T          A32V                      True   
       2358328  T          C          NaN                       True   
       2358646  G          T          NaN                      False   
       2358667  T          A          NaN                      False   
       2358668  G          A          NaN                      False   
...                                                              ...   
       2431005  C          T          R1915R                    True   
       2431061  C          T          A1934V                    True   
       2431079  T          C          I1940T                    True   
       2431194  G          A          Q1978Q                    True   
       2431338  T          C          S2026S                    True   

                                                 pass_gamb_colu  pass_arab  \
contig position ref_allele alt_allele aa_change                              
2L     2358252  C          T          A32V                 True       True   
       2358328  T          C          NaN                  True       True   
       2358646  G          T          NaN                 False       True   
       2358667  T          A          NaN                 False       True   
       2358668  G          A          NaN                 False       True   
...                                                         ...        ...   
       2431005  C          T          R1915R               True       True   
       2431061  C          T          A1934V               True       True   
       2431079  T          C          I1940T               True       True   
       2431194  G          A          Q1978Q               True       True   
       2431338  T          C          S2026S               True       True   

                                                 frq_BF-09_colu_2012  \
contig position ref_allele alt_allele aa_change                        
2L     2358252  C          T          A32V                  0.006098   
       2358328  T          C          NaN                   0.006098   
       2358646  G          T          NaN                   0.000000   
       2358667  T          A          NaN                   0.847561   
       2358668  G          A          NaN                   0.012195   
...                                                              ...   
       2431005  C          T          R1915R                0.865854   
       2431061  C          T          A1934V                0.121951   
       2431079  T          C          I1940T                0.036585   
       2431194  G          A          Q1978Q                0.865854   
       2431338  T          C          S2026S                0.121951   

                                                 frq_BF-09_colu_2014  \
contig position ref_allele alt_allele aa_change                        
2L     2358252  C          T          A32V                  0.000000   
       2358328  T          C          NaN                   0.000000   
       2358646  G          T          NaN                   0.009434   
       2358667  T          A          NaN                   0.877358   
       2358668  G          A          NaN                   0.000000   
...                                                              ...   
       2431005  C          T          R1915R                0.886792   
       2431061  C          T          A1934V                0.132075   
       2431079  T          C          I1940T                0.037736   
       2431194  G          A          Q1978Q                0.886792   
       2431338  T          C          S2026S                0.113208   

                                                   max_af     transcript  \
contig position ref_allele alt_allele aa_change                            
2L     2358252  C          T          A32V       0.006098  AGAP004707-R

## `snp_allele_frequencies_advanced`

The "advanced" counterpart of `snp_allele_frequencies`: instead of a fixed `cohorts` scheme, samples are grouped automatically by taxon, spatial area and temporal period, and the result is an xarray `Dataset` (with confidence intervals) rather than a dataframe. Parameters:

- `transcript` (required): transcript ID to compute frequencies for.
- `area_by` (required): sample-metadata column used to group samples spatially, e.g. `"admin1_iso"` (ISO code of the level-1 administrative division).
- `period_by` (required): either `"year"`, `"quarter"` or `"month"` to bin by calendar period, or the name of a custom period column.
- `sample_sets` / `sample_query` / `sample_query_options`: restrict which samples are used.
- `min_cohort_size`: cohorts smaller than this are simply excluded from the output (default `10`).
- `drop_invariant`: drop variants with zero frequency in every cohort.
- `variant_query`: a pandas query string evaluated against the variant dataframe to further filter which variants are kept, e.g. restricting to non-synonymous changes above some frequency.
- `site_mask`: site-filter mask to apply; `None` applies none.
- `nobs_mode`: `"called"` (default) uses the number of samples with a non-missing genotype call (×2) as the frequency denominator; `"fixed"` uses the cohort size (×2) regardless of missingness.
- `ci_method`: method used to compute frequency confidence intervals (e.g. `"wilson"`), passed to `statsmodels.stats.proportion.proportion_confint`; `None` skips CI computation.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.
- `taxon_by`: sample-metadata column used for taxon stratification; default `"taxon"`.
- `filter_unassigned`: whether to drop samples with intermediate/unassigned taxon calls before grouping; `None` (default) only filters when using the default `"taxon"` column.

In [3]:
snp_freqs_adv_ds = ag3.snp_allele_frequencies_advanced(
    transcript="AGAP004707-RD",
    area_by="admin1_iso",
    period_by="year",
    sample_sets=["AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-UG", "AG1000G-TZ"],
    sample_query="taxon in ['gambiae', 'coluzzii']",
    min_cohort_size=10,
    drop_invariant=True,
    variant_query="max_af > 0.05 and effect == 'NON_SYNONYMOUS_CODING'",
    site_mask=None,
    nobs_mode="called",
    ci_method="wilson",
)
snp_freqs_adv_ds

Load sample metadata: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

Access SNP calls: ⠋ (0:00:00.00)

Access SNP calls: ⠙ (0:00:00.09)

Load SNP genotypes:   0%|          | 0/141 [00:00<?, ?it/s]

Compute SNP allele frequencies:   0%|          | 0/8 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/14677 [00:00<?, ?it/s]

Load genome features: ⠋ (0:00:00.00)

<xarray.Dataset> Size: 8kB
Dimensions:                      (cohorts: 8, variants: 15)
Dimensions without coordinates: cohorts, variants
Data variables: (12/38)
    cohort_area                  (cohorts) object 64B 'BF-09' 'BF-09' ... 'UG-W'
    cohort_label                 (cohorts) object 64B 'BF-09_colu_2012' ... '...
    cohort_lat_max               (cohorts) float64 64B 11.23 11.23 ... -0.751
    cohort_lat_mean              (cohorts) float64 64B 11.22 11.23 ... -0.751
    cohort_lat_min               (cohorts) float64 64B 11.15 11.23 ... -0.751
    cohort_lon_max               (cohorts) float64 64B -4.235 -4.472 ... 29.7
    ...                           ...
    variant_position             (variants) int32 60B 2391228 ... 2431061
    variant_ref_aa               (variants) object 120B 'V' 'V' 'I' ... 'P' 'A'
    variant_ref_allele           (variants) object 120B 'G' 'G' 'A' ... 'C' 'C'
    variant_ref_codon            (variants) object 120B 'Gta' 'Gta' ... 'gCt'
    variant_sneath_score         (variants) float64 120B 7.0 7.0 ... 24.0 12.0
    variant_transcript           (variants) object 120B 'AGAP004707-RD' ... '...
Attributes:
    title:    AGAP004707-RD (Vgsc/para) SNP frequencies

## `aa_allele_frequencies`

A thin wrapper around `snp_allele_frequencies` (called internally with `effects=True`) that further aggregates frequencies **by amino acid change** rather than by individual SNP allele — multiple SNPs causing the same amino acid substitution are combined into one row. Parameters mirror `snp_allele_frequencies` (minus `effects`, which is always on, and minus `region`, since amino-acid annotation requires a transcript):

- `transcript` (required): transcript ID.
- `cohorts` (required): predefined cohort scheme name or custom cohort dict.
- `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size, enforced with a `ValueError` if violated.
- `site_mask`: site-filter mask, or `None` for no filtering.
- `sample_sets`: which sample set(s)/release(s) to use.
- `drop_invariant`: drop amino-acid changes never observed.
- `include_counts`: include raw count/nobs columns.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.

In [4]:
aa_freqs_df = ag3.aa_allele_frequencies(
    transcript="AGAP004707-RD",
    cohorts="admin1_year",
    sample_sets=("AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-BF-C"),
    sample_query="taxon == 'coluzzii'",
)
aa_freqs_df.query("max_af > 0.05")

Load genome features: ⠋ (0:00:00.00)

Load SNP genotypes:   0%|          | 0/57 [00:00<?, ?it/s]

Prepare SNP dataframe: ⠋ (0:00:00.00)

Compute allele frequencies:   0%|          | 0/2 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/5242 [00:00<?, ?it/s]

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

,,,frq_BF-09_colu_2012,frq_BF-09_colu_2014,transcript,aa_pos,ref_allele,ref_aa,alt_aa,effect,impact,grantham_score,sneath_score,alt_allele,max_af,label
aa_change,contig,position,,,,,,,,,,,,,,
V402L,2L,2391228,0.121951,0.113208,AGAP004707-RD,402.0,G,V,L,NON_SYNONYMOUS_CODING,MODERATE,32.0,7.0,"{C,T}",0.121951,"V402L (2L:2,391,228 G>{C,T})"
L995F,2L,2422652,0.865854,0.886792,AGAP004707-RD,995.0,A,L,F,NON_SYNONYMOUS_CODING,MODERATE,22.0,22.0,T,0.886792,"L995F (2L:2,422,652 A>T)"
I1527T,2L,2429617,0.121951,0.113208,AGAP004707-RD,1527.0,T,I,T,NON_SYNONYMOUS_CODING,MODERATE,89.0,23.0,C,0.121951,"I1527T (2L:2,429,617 T>C)"
N1570Y,2L,2429745,0.250000,0.320755,AGAP004707-RD,1570.0,A,N,Y,NON_SYNONYMOUS_CODING,MODERATE,143.0,28.0,T,0.320755,"N1570Y (2L:2,429,745 A>T)"
K1603T,2L,2429915,0.054878,0.056604,AGAP004707-RD,1603.0,A,K,T,NON_SYNONYMOUS_CODING,MODERATE,78.0,34.0,C,0.056604,"K1603T (2L:2,429,915 A>C)"
P1874S,2L,2430880,0.213415,0.169811,AGAP004707-RD,1874.0,C,P,S,NON_SYNONYMOUS_CODING,MODERATE,74.0,24.0,T,0.213415,"P1874S (2L:2,430,880 C>T)"
P1874L,2L,2430881,0.073171,0.056604,AGAP004707-RD,1874.0,C,P,L,NON_SYNONYMOUS_CODING,MODERATE,98.0,24.0,T,0.073171,"P1874L (2L:2,430,881 C>T)"
A1934V,2L,2431061,0.121951,0.132075,AGAP004707-RD,1934.0,C,A,V,NON_SYNONYMOUS_CODING,MODERATE,64.0,12.0,T,0.132075,"A1934V (2L:2,431,061 C>T)"


## `aa_allele_frequencies_advanced`

The "advanced" counterpart of `aa_allele_frequencies`: computes `snp_allele_frequencies_advanced` internally and then aggregates by amino acid change, returning an xarray `Dataset` grouped by taxon/area/period cohorts rather than a fixed `cohorts` scheme. Parameters are the same as `snp_allele_frequencies_advanced` (minus `drop_invariant`, which is not exposed here):

- `transcript`, `area_by`, `period_by` (required).
- `sample_sets` / `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size to include.
- `variant_query`: pandas query to filter the resulting amino-acid-change variants, e.g. by `max_af`.
- `site_mask`: site-filter mask, or `None`.
- `nobs_mode`: `"called"` or `"fixed"` denominator, as above.
- `ci_method`: confidence-interval method.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.
- `taxon_by` / `filter_unassigned`: taxon stratification controls, as above.

In [5]:
aa_freqs_adv_ds = ag3.aa_allele_frequencies_advanced(
    transcript="AGAP004707-RD",
    area_by="admin1_iso",
    period_by="year",
    sample_sets=["AG1000G-BF-A", "AG1000G-BF-B"],
    sample_query="sex_call == 'F' and taxon == 'coluzzii'",
    min_cohort_size=10,
    variant_query="max_af > 0.05",
)
aa_freqs_adv_ds

Load sample metadata: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

Access SNP calls: ⠋ (0:00:00.00)

Access SNP calls: ⠙ (0:00:00.09)

Load SNP genotypes:   0%|          | 0/53 [00:00<?, ?it/s]

Compute SNP allele frequencies:   0%|          | 0/2 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/4686 [00:00<?, ?it/s]

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

<xarray.Dataset> Size: 2kB
Dimensions:                 (cohorts: 2, variants: 10)
Dimensions without coordinates: cohorts, variants
Data variables: (12/31)
    cohort_area             (cohorts) object 16B 'BF-09' 'BF-09'
    cohort_label            (cohorts) object 16B 'BF-09_colu_2012' 'BF-09_col...
    cohort_lat_max          (cohorts) float64 16B 11.23 11.23
    cohort_lat_mean         (cohorts) float64 16B 11.22 11.23
    cohort_lat_min          (cohorts) float64 16B 11.15 11.23
    cohort_lon_max          (cohorts) float64 16B -4.235 -4.472
    ...                      ...
    variant_label           (variants) object 80B 'V402L (2L:2,391,228 G>{C,T...
    variant_max_af          (variants) float64 80B 0.1282 0.875 ... 0.125 0.0625
    variant_position        (variants) int32 40B 2391228 2422652 ... 2431079
    variant_ref_aa          (variants) object 80B 'V' 'L' 'I' ... 'P' 'A' 'I'
    variant_ref_allele      (variants) object 80B 'G' 'A' 'T' ... 'C' 'C' 'T'
    variant_transcript      (variants) object 80B 'AGAP004707-RD' ... 'AGAP00...
Attributes:
    title:    AGAP004707-RD (Vgsc/para) SNP frequencies

## `gene_cnv_frequencies`

Computes modal gene copy number (via `gene_cnv`) and then, per cohort, the frequency of amplification (`amp`, CN above the diploid expectation) and deletion (`del`, CN below expectation) — one row per gene per CNV type. Parameters:

- `region` (required): genome region(s), or a list of gene IDs, to compute gene CNV frequencies for.
- `cohorts` (required): predefined cohort scheme name or custom cohort dict.
- `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size (default `10`); cohorts below this are excluded.
- `max_coverage_variance`: samples with coverage variance above this threshold are excluded before computing modal CN (default `0.2`); `None` keeps all samples.
- `sample_sets`: which sample set(s)/release(s) to use.
- `drop_invariant`: drop gene/CNV-type rows with zero frequency everywhere.
- `include_counts`: include raw amp/del count and `nobs` columns.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.

The example below uses the Cyp6aa/p amplification hotspot region on 2R.

In [6]:
cyp6aap_region = "2R:28,450,000-28,510,000"

gene_cnv_freqs_df = ag3.gene_cnv_frequencies(
    region=cyp6aap_region,
    cohorts="admin1_year",
    sample_sets=("AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-BF-C"),
    sample_query="taxon == 'coluzzii'",
)
gene_cnv_freqs_df

Load genome features: ⠋ (0:00:00.00)

Access CNV HMM data: ⠋ (0:00:00.00)

Access CNV HMM data: ⠙ (0:00:00.09)

Access CNV HMM data: ⠹ (0:00:00.18)

Access CNV HMM data: ⠸ (0:00:00.27)

...

Access CNV HMM data: ⠏ (0:00:03.34)

Access CNV HMM data: ⠋ (0:00:03.42)

Load CNV HMM data:   0%|          | 0/39 [00:00<?, ?it/s]

Compute modal gene copy number:   0%|          | 0/11 [00:00<?, ?it/s]

,,,gene_strand,gene_description,contig,start,end,frq_BF-09_colu_2012,frq_BF-09_colu_2014,max_af,windows,label
gene_id,gene_name,cnv_type,,,,,,,,,,
AGAP002862,CYP6AA1,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28480576,28482637,0.9125,0.811321,0.912500,8,AGAP002862 (CYP6AA1) amp
AGAP013128,CYP6AA2,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28483301,28484921,0.8250,0.792453,0.825000,6,AGAP013128 (CYP6AA2) amp
AGAP002863,COEAE6O,amp,-,carboxylesterase alpha esterase [Source:VB Com...,2R,28485262,28487080,0.6000,0.509434,0.600000,7,AGAP002863 (COEAE6O) amp
AGAP002864,CYP6P15P,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28487640,28489092,0.5250,0.528302,0.528302,6,AGAP002864 (CYP6P15P) amp
AGAP002865,CYP6P3,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28491415,28493141,0.0375,0.075472,0.075472,7,AGAP002865 (CYP6P3) amp
AGAP002866,CYP6P5,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28494017,28495645,0.0375,0.056604,0.056604,6,AGAP002866 (CYP6P5) amp
AGAP002867,CYP6P4,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28497087,28498674,0.0375,0.075472,0.075472,6,AGAP002867 (CYP6P4) amp
AGAP002868,CYP6P1,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28499251,28500900,0.0375,0.056604,0.056604,6,AGAP002868 (CYP6P1) amp
AGAP002869,CYP6P2,amp,-,cytochrome P450 [Source:VB Community Annotation],2R,28501033,28502910,0.0375,0.056604,0.056604,7,AGAP002869 (CYP6P2) amp


## `gene_cnv_frequencies_advanced`

The "advanced" counterpart of `gene_cnv_frequencies`: groups samples by taxon/area/period cohorts and returns an xarray `Dataset` of amplification/deletion counts and frequencies with cohort metadata and confidence intervals. Parameters:

- `region` (required), `area_by` (required), `period_by` (required): as above.
- `sample_sets` / `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size to include (default `10`).
- `drop_invariant`: drop invariant gene/CNV-type rows.
- `variant_query`: pandas query to further filter the resulting gene/CNV-type rows.
- `max_coverage_variance`: coverage-variance sample filter, as above.
- `nobs_mode` / `ci_method`: denominator mode and confidence-interval method, as above.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.
- `taxon_by` / `filter_unassigned`: taxon stratification controls.

In [7]:
gene_cnv_freqs_adv_ds = ag3.gene_cnv_frequencies_advanced(
    region=cyp6aap_region,
    area_by="admin1_iso",
    period_by="year",
    sample_sets=["AG1000G-BF-A", "AG1000G-BF-B"],
    sample_query="taxon in ['coluzzii', 'gambiae']",
    min_cohort_size=10,
)
gene_cnv_freqs_adv_ds

Load genome features: ⠋ (0:00:00.00)

Access CNV HMM data: ⠋ (0:00:00.00)

Load CNV HMM data:   0%|          | 0/48 [00:00<?, ?it/s]

Compute modal gene copy number:   0%|          | 0/11 [00:00<?, ?it/s]

<xarray.Dataset> Size: 3kB
Dimensions:                 (cohorts: 4, variants: 11)
Dimensions without coordinates: cohorts, variants
Data variables: (12/28)
    cohort_area             (cohorts) object 32B 'BF-09' 'BF-09' 'BF-09' 'BF-09'
    cohort_label            (cohorts) object 32B 'BF-09_colu_2012' ... 'BF-09...
    cohort_lat_max          (cohorts) float64 32B 11.23 11.23 11.23 11.23
    cohort_lat_mean         (cohorts) float64 32B 11.22 11.23 11.19 11.21
    cohort_lat_min          (cohorts) float64 32B 11.15 11.23 11.15 11.15
    cohort_lon_max          (cohorts) float64 32B -4.235 -4.472 -4.235 -4.235
    ...                      ...
    variant_gene_name       (variants) object 88B 'CYP6AA1' ... 'CYP6AD1'
    variant_gene_strand     (variants) object 88B '-' '-' '-' ... '-' '-' '-'
    variant_label           (variants) object 88B 'AGAP002862 (CYP6AA1) amp' ...
    variant_max_af          (variants) float64 88B 0.9125 0.825 ... 0.0566
    variant_start           (variants) int64 88B 28480576 28483301 ... 28504248
    variant_windows         (variants) int64 88B 8 6 7 6 7 7 6 6 6 7 6
Attributes:
    title:    Gene CNV frequencies (2R:28,450,000-28,510,000)

## `haplotypes_frequencies`

Computes frequencies of distinct **phased haplotypes** over a region, across cohorts, returning one row per distinct haplotype. Parameters:

- `region` (required): genome region over which haplotypes are defined (a narrower region gives fewer, more common distinct haplotypes; a wider region gives more, rarer ones).
- `cohorts` (required): predefined cohort scheme name or custom cohort dict.
- `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size (default `10`).
- `sample_sets`: which sample set(s)/release(s) to use.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.

The example below uses a ~73 kbp window around the *Vgsc* gene.

In [8]:
hap_freqs_df = ag3.haplotypes_frequencies(
    region="2L:2,358,158-2,431,617",
    cohorts="admin1_year",
    sample_sets=("AG1000G-BF-A", "AG1000G-BF-B", "AG1000G-BF-C"),
)
hap_freqs_df.query("max_af > .05")

Access haplotypes: ⠋ (0:00:00.00)

Access haplotypes: ⠙ (0:00:00.08)

Access haplotypes: ⠹ (0:00:00.17)

Access haplotypes: ⠸ (0:00:00.25)

Access haplotypes: ⠼ (0:00:00.34)

...

Access haplotypes: ⠴ (0:00:19.15)

Access haplotypes: ⠦ (0:00:19.23)

Compute haplotypes:   0%|          | 0/13 [00:00<?, ?it/s]

Compute allele frequencies:   0%|          | 0/5 [00:00<?, ?it/s]

,frq_BF-07_gamb_2004,frq_BF-09_colu_2012,frq_BF-09_colu_2014,frq_BF-09_gamb_2012,frq_BF-09_gamb_2014,max_af
label,,,,,,
H0,0.038462,0.146341,0.169811,0.090909,0.054348,0.169811
H1,0.000000,0.121951,0.094340,0.000000,0.010870,0.121951
H2,0.000000,0.006098,0.028302,0.055556,0.119565,0.119565
H3,0.000000,0.042683,0.028302,0.090909,0.108696,0.108696
H4,0.000000,0.000000,0.009434,0.090909,0.086957,0.090909
H5,0.000000,0.000000,0.009434,0.070707,0.086957,0.086957
H6,0.000000,0.079268,0.047170,0.005051,0.021739,0.079268
H7,0.000000,0.000000,0.000000,0.045455,0.076087,0.076087
H8,0.000000,0.048780,0.056604,0.075758,0.076087,0.076087


## `haplotypes_frequencies_advanced`

The "advanced" counterpart of `haplotypes_frequencies`: groups samples by taxon/area/period cohorts and returns an xarray `Dataset` of per-haplotype counts, frequencies, and confidence intervals with cohort metadata. Parameters:

- `region` (required), `area_by` (required), `period_by` (required): as above.
- `sample_sets` / `sample_query` / `sample_query_options`: restrict samples used.
- `min_cohort_size`: minimum cohort size to include.
- `ci_method`: confidence-interval method.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.
- `taxon_by` / `filter_unassigned`: taxon stratification controls.

In [9]:
hap_freqs_adv_ds = ag3.haplotypes_frequencies_advanced(
    region="2L:2,358,158-2,431,617",
    area_by="admin1_iso",
    period_by="year",
    sample_sets=["AG1000G-BF-A", "AG1000G-BF-B"],
)
hap_freqs_adv_ds

Access haplotypes: ⠋ (0:00:00.00)

Compute haplotypes:   0%|          | 0/11 [00:00<?, ?it/s]

Compute allele frequencies: 0it [00:00, ?it/s]

<xarray.Dataset> Size: 30kB
Dimensions:                 (cohorts: 4, variants: 175)
Dimensions without coordinates: cohorts, variants
Data variables: (12/19)
    cohort_taxon            (cohorts) object 32B 'coluzzii' ... 'gambiae'
    cohort_area             (cohorts) object 32B 'BF-09' 'BF-09' 'BF-09' 'BF-09'
    cohort_period           (cohorts) period[Y-DEC] 32B 2012 2014 2012 2014
    cohort_size             (cohorts) int64 32B 82 53 99 46
    cohort_lat_mean         (cohorts) float64 32B 11.22 11.23 11.19 11.21
    cohort_lat_max          (cohorts) float64 32B 11.23 11.23 11.23 11.23
    ...                      ...
    variant_label           (variants) object 1kB 'H0' 'H1' ... 'H173' 'H174'
    event_frequency         (variants, cohorts) object 6kB 0.1463414634146341...
    event_count             (variants, cohorts) object 6kB 24 18 18 5 ... 0 0 0
    event_nobs              (variants, cohorts) object 6kB 164 106 ... 198 92
    event_frequency_ci_low  (variants, cohorts) float64 6kB 0.1004 ... 0.0
    event_frequency_ci_upp  (variants, cohorts) float64 6kB 0.2085 ... 0.04008

## `plot_frequencies_heatmap`

Plots a **cohort × variant** frequency dataframe (from any of the simple `*_frequencies` methods above) as a heatmap, using plotly. Parameters:

- `df` (required): a frequencies dataframe as returned by `snp_allele_frequencies`, `aa_allele_frequencies`, `gene_cnv_frequencies` or `haplotypes_frequencies`.
- `index`: column(s) used to label heatmap rows; default `"label"`. `None` falls back to using the dataframe's index columns.
- `max_len`: safety limit on the number of rows plotted (default `100`); exceeding it raises a `ValueError` unless set to `None`, to avoid accidentally rendering an enormous heatmap.
- `col_width` / `row_height`: pixel size per column/row, used to size the figure.
- `x_label` / `y_label`: axis titles.
- `colorbar`: if `False`, hide the color bar.
- `width` / `height`: explicit figure size in pixels; `None` sizes automatically from the data shape.
- `text_auto`: if truthy, overlay numeric text on each cell; a format string (default `".0%"`) controls how values are formatted (here, as a percentage).
- `aspect`: `"auto"` or `"equal"` cell aspect ratio.
- `color_continuous_scale`: name of (or list of) colours for the frequency colour scale (default `"Reds"`).
- `title`: if `True` (default), use the dataframe's `attrs['title']` if present; or supply a custom string.
- `show` / `renderer`: whether to immediately display the figure, and which plotly renderer to use.

Here we plot non-synonymous SNP frequencies for *Vgsc* with a custom title.

In [10]:
nonsyn_df = snp_freqs_df.query("effect == 'NON_SYNONYMOUS_CODING' and max_af > 0.05")
ag3.plot_frequencies_heatmap(df=nonsyn_df, title="Vgsc non-synonymous SNP frequencies")

## `plot_frequencies_time_series`

Plots frequencies over time as a line chart (with confidence-interval shading, if present), using an xarray `Dataset` from any of the `*_advanced` methods above (one line per variant/cohort-area combination). Parameters:

- `ds` (required): a frequencies `Dataset` from `snp_allele_frequencies_advanced`, `aa_allele_frequencies_advanced`, `gene_cnv_frequencies_advanced` or `haplotypes_frequencies_advanced`.
- `height` / `width`: figure size in pixels; `None` sizes automatically.
- `title`: if `True` (default), use `ds.attrs['title']`; or a custom string.
- `legend_sizing`: plotly legend item-sizing mode (e.g. `"constant"`).
- `show` / `renderer`: whether to display immediately, and which plotly renderer to use.
- `taxa`: restrict the plot to one taxon or a list of taxa; `None` (default) plots all.
- `areas`: restrict the plot to one area or a list of areas; `None` (default) plots all.

Below we reuse the gene CNV frequencies advanced dataset and restrict to the `coluzzii` taxon.

In [11]:
ag3.plot_frequencies_time_series(
    gene_cnv_freqs_adv_ds,
    taxa="coluzzii",
    height=500,
    width=1000,
)

## `plot_frequencies_interactive_map`

Builds an interactive ipyleaflet map with dropdown widgets (variant, taxon, period) that lets you browse how a variant's frequency varies spatially for a chosen taxon/period, by calling `plot_frequencies_map_markers` internally whenever a dropdown changes. Parameters:

- `ds` (required): a frequencies `Dataset` from one of the `*_advanced` methods.
- `center`: initial map centre as `(lat, lon)`; default `(-2, 20)` (central Africa).
- `zoom`: initial zoom level; default `3`.
- `title`: if `True` (default), use `ds.attrs['title']` as a heading above the map; `False`/`None` omits it, or supply a custom string.
- `epilogue`: if `True` (default), show a standard explanatory caption below the map; `False` omits it, or supply custom text.

In [12]:
ag3.plot_frequencies_interactive_map(snp_freqs_adv_ds)

## `plot_frequencies_map_markers`

The lower-level function that `plot_frequencies_interactive_map` wraps: given an existing ipyleaflet map, draws one circle marker per cohort at its mean lat/lon, coloured by frequency, for a single chosen variant/taxon/period combination. Useful for embedding a static frequency-map snapshot without the interactive dropdown controls. Parameters:

- `m` (required): an existing `ipyleaflet.Map` instance to draw markers on.
- `ds` (required): a frequencies `Dataset` from one of the `*_advanced` methods.
- `variant` (required): which variant to show, either its integer position in the `variants` dimension or its string `variant_label`.
- `taxon` (required): which taxon's cohorts to show markers for.
- `period` (required): which time period's cohorts to show markers for (a `pandas.Period`, e.g. as found in `ds['cohort_period']`).
- `clear`: if `True` (default), remove any existing marker layers from the map before adding new ones; `False` adds markers on top of whatever is already there.

In [13]:
import ipyleaflet

m = ipyleaflet.Map(center=(12, -2), zoom=6)

first_taxon = gene_cnv_freqs_adv_ds["cohort_taxon"].values[0]
first_period = gene_cnv_freqs_adv_ds["cohort_period"].values[0]
first_variant = gene_cnv_freqs_adv_ds["variant_label"].values[0]

ag3.plot_frequencies_map_markers(
    m=m,
    ds=gene_cnv_freqs_adv_ds,
    variant=first_variant,
    taxon=first_taxon,
    period=first_period,
)
m

Map(center=[12, -2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…